# DLinear — Walmart Store Sales Forecasting

## DLinear არქიტექტურა

**Decomposition Linear (2023)** — Time Series-ს ყოფს ორ კომპონენტად და თითოეულს Linear Layer-ით ამუშავებს:

- **Trend** = Moving Average of input
- **Seasonal** = Input − Trend
- **Output** = Linear(Trend) + Linear(Seasonal)

**WandB Runs:** `DLinear_Cleaning` → `DLinear_Feature_Engineering` → `DLinear_Baseline` → `DLinear_Tuned` → `DLinear_Best_Pipeline`

## ბიბლიოთეკები და Device

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import wandb
import pickle
import os
import subprocess
import warnings
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## WandB კონფიგურაცია

In [ ]:
subprocess.run(['rm', '-f', '/root/.netrc'], capture_output=True)

os.environ["WANDB_API_KEY"] = "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"
wandb.login(key=os.environ["WANDB_API_KEY"], relogin=True)

WANDB_PROJECT = 'walmart-sales-forecasting-project'
WANDB_ENTITY  = 'ashos22-free-university-of-tbilisi-'
print('WandB login OK')

## მეტრიკა და Hyperparameter-ები

In [ ]:
SEQ_LEN  = 52
PRED_LEN = 39

def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)

## მონაცემების ჩატვირთვა

In [ ]:
DATA_PATH = '/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/'

train    = pd.read_csv(DATA_PATH + 'train.csv.zip')
test     = pd.read_csv(DATA_PATH + 'test.csv.zip')
stores   = pd.read_csv(DATA_PATH + 'stores.csv')
features = pd.read_csv(DATA_PATH + 'features.csv.zip')

print('train:   ', train.shape)
print('test:    ', test.shape)
print('stores:  ', stores.shape)
print('features:', features.shape)
train.head()

## WandB Run — DLinear_Cleaning

In [ ]:
run = wandb.init(
    project=WANDB_PROJECT, entity=WANDB_ENTITY,
    name='DLinear_Cleaning', group='DLinear_Training', reinit=True
)

for df in [train, test, features]:
    df['Date'] = pd.to_datetime(df['Date'])

train = train.merge(stores, on='Store', how='left')
train = train.merge(features, on=['Store', 'Date'], how='left', suffixes=('', '_feat'))
test  = test.merge(stores, on='Store', how='left')
test  = test.merge(features, on=['Store', 'Date'], how='left', suffixes=('', '_feat'))

for df in [train, test]:
    if 'IsHoliday_feat' in df.columns:
        df.drop('IsHoliday_feat', axis=1, inplace=True)

markdown_cols = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
for df in [train, test]:
    df[markdown_cols] = df[markdown_cols].fillna(0)

for col in ['CPI', 'Unemployment']:
    train[col] = train.groupby('Store')[col].transform(lambda x: x.ffill())
    test[col]  = test.groupby('Store')[col].transform(lambda x: x.ffill())

train['IsHoliday'] = train['IsHoliday'].astype(int)
test['IsHoliday']  = test['IsHoliday'].astype(int)

wandb.log({
    'train_rows': len(train),
    'test_rows':  len(test),
    'null_train': int(train.isnull().sum().sum())
})
run.finish()
print('გასუფთავება დასრულდა')

## WandB Run — DLinear_Feature_Engineering

თითოეული (Store, Dept) წყვილისთვის ცალ-ცალკე Time Series + normalization.

In [ ]:
run = wandb.init(
    project=WANDB_PROJECT, entity=WANDB_ENTITY,
    name='DLinear_Feature_Engineering', group='DLinear_Training', reinit=True
)

series_info = {}
for (store, dept), group in train.groupby(['Store', 'Dept']):
    group = group.sort_values('Date')
    sales      = group['Weekly_Sales'].values.astype(np.float32)
    is_holiday = group['IsHoliday'].values.astype(np.float32)
    mean = float(sales.mean())
    std  = float(sales.std()) + 1e-8
    series_info[(store, dept)] = {
        'sales':      sales,
        'is_holiday': is_holiday,
        'dates':      group['Date'].values,
        'mean':       mean,
        'std':        std
    }

test_dates  = sorted(test['Date'].unique())
date_to_idx = {d: i for i, d in enumerate(test_dates)}

n_series = len(series_info)
avg_len  = np.mean([len(v['sales']) for v in series_info.values()])

wandb.config.update({
    'n_series':       n_series,
    'avg_series_len': round(avg_len, 1),
    'seq_len':        SEQ_LEN,
    'pred_len':       PRED_LEN,
    'n_test_dates':   len(test_dates)
})
run.finish()

print(f'სერიების რაოდენობა: {n_series}')
print(f'საშუალო სიგრძე:    {avg_len:.1f} კვირა')
print(f'Test dates:         {len(test_dates)}')

## DLinear მოდელი

In [ ]:
class MovingAvg(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.kernel_size = kernel_size
        self.avg = nn.AvgPool1d(kernel_size=kernel_size, stride=1)

    def forward(self, x):
        pad   = self.kernel_size // 2
        front = x[:, :1, :].repeat(1, pad, 1)
        end   = x[:, -1:, :].repeat(1, pad, 1)
        x     = torch.cat([front, x, end], dim=1)
        x     = self.avg(x.permute(0, 2, 1)).permute(0, 2, 1)
        return x


class SeriesDecomp(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.moving_avg = MovingAvg(kernel_size)

    def forward(self, x):
        trend    = self.moving_avg(x)
        seasonal = x - trend
        return seasonal, trend


class DLinear(nn.Module):
    def __init__(self, seq_len, pred_len, kernel_size=13):
        super().__init__()
        self.decomp          = SeriesDecomp(kernel_size)
        self.linear_trend    = nn.Linear(seq_len, pred_len)
        self.linear_seasonal = nn.Linear(seq_len, pred_len)

    def forward(self, x):
        seasonal, trend = self.decomp(x)
        t = self.linear_trend(trend.permute(0, 2, 1)).permute(0, 2, 1)
        s = self.linear_seasonal(seasonal.permute(0, 2, 1)).permute(0, 2, 1)
        return t + s


print('DLinear კლასები მზადაა')

## Dataset და Dataset-ის ფუნქციები

In [ ]:
class WalmartDataset(Dataset):
    def __init__(self, series_info, seq_len, pred_len, split='train'):
        self.X, self.y = [], []

        for key, info in series_info.items():
            sales = info['sales']
            n     = len(sales)
            if n < seq_len + 2 * pred_len:
                continue

            mean = info['mean']
            std  = info['std']
            norm = (sales - mean) / std

            val_start = n - pred_len

            if split == 'train':
                for i in range(seq_len, val_start - pred_len + 1):
                    self.X.append(norm[i - seq_len:i])
                    self.y.append(norm[i:i + pred_len])
            else:
                if val_start >= seq_len:
                    self.X.append(norm[val_start - seq_len:val_start])
                    self.y.append(norm[val_start:n])

        self.X = torch.FloatTensor(np.array(self.X)).unsqueeze(-1)
        self.y = torch.FloatTensor(np.array(self.y)).unsqueeze(-1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def compute_val_wmae(model, series_info, seq_len, pred_len):
    model.eval()
    all_true, all_pred, all_hol = [], [], []

    with torch.no_grad():
        for key, info in series_info.items():
            sales      = info['sales']
            is_holiday = info['is_holiday']
            mean, std  = info['mean'], info['std']
            n          = len(sales)

            val_start = n - pred_len
            if val_start < seq_len:
                continue

            norm = (sales - mean) / std
            x    = torch.FloatTensor(norm[val_start - seq_len:val_start]).unsqueeze(0).unsqueeze(-1).to(DEVICE)

            pred = model(x).squeeze().cpu().numpy()
            pred = np.maximum(pred * std + mean, 0)

            all_pred.extend(pred)
            all_true.extend(sales[val_start:val_start + pred_len])
            all_hol.extend(is_holiday[val_start:val_start + pred_len])

    return wmae(np.array(all_true), np.array(all_pred), np.array(all_hol))


print('Dataset კლასი მზადაა')

## სატრენინგო ფუნქცია

In [ ]:
def train_dlinear(params, run_name):
    model = DLinear(
        seq_len=SEQ_LEN,
        pred_len=PRED_LEN,
        kernel_size=params['kernel_size']
    ).to(DEVICE)

    train_ds = WalmartDataset(series_info, SEQ_LEN, PRED_LEN, split='train')
    val_ds   = WalmartDataset(series_info, SEQ_LEN, PRED_LEN, split='val')

    train_loader = DataLoader(train_ds, batch_size=params['batch_size'], shuffle=True,  num_workers=2)
    val_loader   = DataLoader(val_ds,   batch_size=params['batch_size'], shuffle=False, num_workers=2)

    optimizer = torch.optim.Adam(model.parameters(), lr=params['lr'])
    criterion = nn.MSELoss()

    run = wandb.init(
        project=WANDB_PROJECT, entity=WANDB_ENTITY,
        name=run_name, group='DLinear_Training',
        config=params, reinit=True
    )

    best_val_loss    = float('inf')
    best_state       = None
    patience_counter = 0

    for epoch in range(params['epochs']):
        model.train()
        tr_losses = []
        for X, y in train_loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(X), y)
            loss.backward()
            optimizer.step()
            tr_losses.append(loss.item())

        model.eval()
        vl_losses = []
        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(DEVICE), y.to(DEVICE)
                vl_losses.append(criterion(model(X), y).item())

        tr_loss = np.mean(tr_losses)
        vl_loss = np.mean(vl_losses)
        wandb.log({'train_loss': tr_loss, 'val_loss': vl_loss, 'epoch': epoch + 1})

        if vl_loss < best_val_loss:
            best_val_loss    = vl_loss
            best_state       = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= params['patience']:
            print(f'Early stop: epoch {epoch + 1}')
            break

        if (epoch + 1) % 5 == 0:
            print(f'Epoch {epoch+1:3d} | Train: {tr_loss:.4f} | Val: {vl_loss:.4f}')

    model.load_state_dict(best_state)

    val_wmae_score = compute_val_wmae(model, series_info, SEQ_LEN, PRED_LEN)
    wandb.log({'best_val_loss': best_val_loss, 'wmae_val': val_wmae_score})
    run.finish()

    print(f'{run_name} → Best Val Loss: {best_val_loss:.4f} | WMAE: {val_wmae_score:.4f}')
    return model, best_val_loss, val_wmae_score


print('train_dlinear ფუნქცია მზადაა')

## WandB Run — DLinear_Baseline

In [ ]:
params_baseline = {
    'kernel_size': 13,
    'lr':          0.001,
    'batch_size':  256,
    'epochs':      40,
    'patience':    7
}

model_baseline, loss_bl, wmae_bl = train_dlinear(params_baseline, 'DLinear_Baseline')

## WandB Run — DLinear_Tuned

Larger kernel (ტენდენცია უფრო გლუვად) + smaller lr.

In [ ]:
params_tuned = {
    'kernel_size': 25,
    'lr':          0.0005,
    'batch_size':  128,
    'epochs':      60,
    'patience':    10
}

model_tuned, loss_tuned, wmae_tuned = train_dlinear(params_tuned, 'DLinear_Tuned')

## შედეგების შედარება

In [ ]:
scores = {'Baseline': (model_baseline, params_baseline, wmae_bl),
          'Tuned':    (model_tuned,    params_tuned,    wmae_tuned)}

best_name = min(scores, key=lambda k: scores[k][2])
best_model, best_params, best_wmae = scores[best_name]

for k, (_, _, w) in scores.items():
    marker = ' ← საუკეთესო' if k == best_name else ''
    print(f'  {k}: WMAE = {w:.4f}{marker}')

## Test Set-ზე პროგნოზი

In [ ]:
def predict_test(model, series_info, test_df, seq_len, pred_len):
    t_dates     = sorted(test_df['Date'].unique())
    date_to_idx = {d: i for i, d in enumerate(t_dates)}

    model.eval()
    results = []

    with torch.no_grad():
        for (store, dept), info in series_info.items():
            mask = (test_df['Store'] == store) & (test_df['Dept'] == dept)
            rows = test_df[mask].sort_values('Date')
            if len(rows) == 0:
                continue

            sales      = info['sales']
            mean, std  = info['mean'], info['std']

            if len(sales) >= seq_len:
                last = sales[-seq_len:]
            else:
                pad  = np.full(seq_len - len(sales), mean, dtype=np.float32)
                last = np.concatenate([pad, sales])

            norm = (last - mean) / std
            x    = torch.FloatTensor(norm).unsqueeze(0).unsqueeze(-1).to(DEVICE)

            pred = model(x).squeeze().cpu().numpy()
            pred = np.maximum(pred * std + mean, 0)

            for _, row in rows.iterrows():
                idx = date_to_idx.get(row['Date'], 0)
                results.append({
                    'Store': store, 'Dept': dept,
                    'Date':  row['Date'],
                    'Weekly_Sales': pred[idx] if idx < len(pred) else float(mean)
                })

    return pd.DataFrame(results)


test_preds = predict_test(best_model, series_info, test, SEQ_LEN, PRED_LEN)
print(f'პროგნოზი: {len(test_preds):,} rows')
test_preds.head()

## WandB Artifact — DLinear_Best_Pipeline

In [ ]:
run = wandb.init(
    project=WANDB_PROJECT, entity=WANDB_ENTITY,
    name='DLinear_Best_Pipeline', group='DLinear_Training', reinit=True
)

torch.save(best_model.state_dict(), 'dlinear_model.pt')

with open('series_info.pkl', 'wb') as f:
    pickle.dump(series_info, f)

with open('dlinear_config.pkl', 'wb') as f:
    pickle.dump({
        'seq_len':     SEQ_LEN,
        'pred_len':    PRED_LEN,
        'kernel_size': best_params['kernel_size']
    }, f)

artifact = wandb.Artifact(
    name='dlinear-walmart-sales',
    type='model',
    metadata={'wmae_val': best_wmae, 'best_version': best_name}
)
artifact.add_file('dlinear_model.pt')
artifact.add_file('series_info.pkl')
artifact.add_file('dlinear_config.pkl')
run.log_artifact(artifact)

wandb.log({'wmae_val_best': best_wmae})
run.finish()

print(f'WandB Artifact: dlinear-walmart-sales')
print(f'საუკეთესო WMAE: {best_wmae:.4f} ({best_name})')